# Rozdział 4. Od pojedynczej realizacji do właściwości procesu

Notebook demonstracyjny do `manuscript/chapters/ch04_psd_korelacja_usrednianie.md`.
Eksperymenty mają stabilne identyfikatory E4.1–E4.12. Wywód, założenia i ograniczenia
znajdują się w rozdziale; tutaj utrwalono parametry, generatory, obliczenia i kontrole.

Wszystkie dane są syntetyczne. Przykłady EEG-podobne i odpowiedzi wywołanej nie
zawierają danych pacjentów. Nie wymagają pobierania plików ani dostępu do sieci.
Źródło główne: kanoniczny materiał M3; notebook kursowy M3 wykorzystano pomocniczo.

Uruchomienie od początku: Python 3.12, zależności z `requirements_ch04.txt`, katalog
bieżący w katalogu pakietu albo `notebooks/`. Funkcje wspólne znajdują się w `src/bspsbook`.
Komórki zapisują figury jako osobne PNG i SVG, bez osadzania obrazów w base64.
Alternatywne wykonanie wszystkich komórek: `python scripts/run_ch04.py`.


In [1]:
from pathlib import Path
import sys, json, platform
import numpy as np
import scipy
from scipy import signal
import matplotlib
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p/'src/bspsbook').is_dir()), None)
if ROOT is None:
    raise RuntimeError("Uruchom notebook z katalogu pakietu lub notebooks/.")
sys.path.insert(0, str(ROOT/'src'))
from bspsbook.plotting import configure_plots, save_figure
from bspsbook.spectral import (periodogram_density, segment_starts, welch_manual,
                               autocorrelation, cross_correlation, one_sided)
from bspsbook.signals import (colored_background, pulse_train, evoked_response, delay_zeros)
configure_plots(chapter=4)
FIG = ROOT/'figures/generated'
RESULTS = {}
CHECKS = []
BASE_SEED = 20261031
def check(name, condition):
    if not bool(condition):
        raise AssertionError(name)
    CHECKS.append(name)
    print("OK:", name)

ENV = {"python": platform.python_version(), "numpy": np.__version__,
       "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
       "base_seed": BASE_SEED, "rng": "numpy.random.default_rng / PCG64"}
print(json.dumps(ENV, ensure_ascii=False, indent=2))


{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "scipy": "1.17.0",
  "matplotlib": "3.10.8",
  "base_seed": 20261031,
  "rng": "numpy.random.default_rng / PCG64"
}


## E4.1 — realizacja a proces; rys. 4.1

`fs=200 Hz`, `T=4 s`, sześć realizacji, `A=1 j.u.`, `f0=18.3 Hz`,
`SD szumu=1.35 j.u.`; faza jednostajna i niezależne próbki normalne.
Periodogramy: okresowe okno Hanna, odjęcie średniej, PSD jednostronna, `nfft=N`.
Seed: `BASE_SEED+1`.

![Rys. 4.1](../figures/generated/fig_04_01_realizations.png)


In [2]:
rng = np.random.default_rng(BASE_SEED+1)
fs, duration = 200.0, 4.0
t = np.arange(round(fs*duration))/fs
phases = rng.uniform(0, 2*np.pi, 6)
realizations = np.cos(2*np.pi*18.3*t[None,:]+phases[:,None])
realizations += rng.normal(0, 1.35, realizations.shape)
w = signal.get_window('hann', t.size, fftbins=True)
f, ps = signal.periodogram(realizations, fs=fs, window=w, detrend='constant',
                           scaling='density', return_onesided=True, axis=-1)
fig, ax = plt.subplots(2, 1, figsize=(8.8, 6.0), layout='constrained')
for r, ls in [(0, '-'), (1, '--')]:
    ax[0].plot(t, realizations[r], ls, lw=1.0, label=f'Realizacja {r+1}')
ax[0].set(xlim=(0,.5), xlabel='Czas [s]', ylabel='Amplituda [j.u.]', title='a) Ten sam model, różne przebiegi')
ax[0].legend()
for r in range(6):
    ax[1].semilogy(f, ps[r], lw=.65, alpha=.5, color='#727272')
ax[1].semilogy(f, ps.mean(axis=0), lw=1.8, color='#176A8A', label='Średnia 6 periodogramów')
ax[1].axvline(18.3, color='black', ls=':', lw=1, label='f₀ = 18,3 Hz')
ax[1].set(xlim=(2,45), ylim=(1e-4,5), xlabel='Częstotliwość [Hz]', ylabel='PSD [j.u.²/Hz]', title='b) Powtarzalne maksimum i zmienne tło')
ax[1].legend()
peak_freq = f[(f>=10)&(f<=30)][np.argmax(ps[:,(f>=10)&(f<=30)], axis=1)]
check('E4.1: maksimum każdej realizacji w otoczeniu 18,3 Hz', np.all(np.abs(peak_freq-18.3)<.6))
RESULTS['E4.1'] = {'peaks_Hz': peak_freq.tolist(), 'N':t.size, 'fs_Hz':fs}
save_figure(fig, FIG, 'fig_04_01_realizations')


OK: E4.1: maksimum każdej realizacji w otoczeniu 18,3 Hz
Figura: figures/generated/fig_04_01_realizations.png (+ SVG)


## E4.2 — rozkład estymatora; rys. 4.2

2000 powtórzeń dla każdej długości `N=25,100,400`; niezależne próbki normalne,
średnia `2 j.u.`, SD `3 j.u.`. Wariancja z dzielnikiem N−1. Seed: `BASE_SEED+2`.
Oddzielne jednostki obu paneli wykluczają porównanie ich szerokości bez normalizacji.

![Rys. 4.2](../figures/generated/fig_04_02_estimators.png)


In [3]:
rng = np.random.default_rng(BASE_SEED+2)
fig, ax = plt.subplots(1,2,figsize=(9.2,3.8),layout='constrained')
mc_rows = []
colors = ['#176A8A','#B25324','#25805D']
for N, color in zip([25,100,400],colors):
    draws = rng.normal(2,3,(2000,N))
    means, variances = draws.mean(axis=1), draws.var(axis=1,ddof=1)
    ax[0].hist(means,bins=45,density=True,histtype='step',color=color,label=f'N={N}',lw=1.5)
    ax[1].hist(variances,bins=45,density=True,histtype='step',color=color,label=f'N={N}',lw=1.5)
    sd = means.std(ddof=1)
    check(f'E4.2: SD średniej dla N={N} zgodne z σ/√N (8%)', abs(sd/(3/np.sqrt(N))-1)<.08)
    check(f'E4.2: średnia estymat wariancji N={N} (5%)', abs(variances.mean()/9-1)<.05)
    mc_rows.append({'N':N,'mean_estimate':float(means.mean()),'SD_mean':float(sd),
                    'mean_variance':float(variances.mean()),'theory_SD_mean':float(3/np.sqrt(N))})
ax[0].axvline(2,color='black',ls='--',lw=1)
ax[1].axvline(9,color='black',ls='--',lw=1)
ax[0].set(xlabel='Estymata średniej [j.u.]',ylabel='Gęstość [1/j.u.]',title='a) Średnia')
ax[1].set(xlabel='Estymata wariancji [j.u.²]',ylabel='Gęstość [1/j.u.²]',title='b) Wariancja, dzielnik N−1')
for a in ax:a.legend()
RESULTS['E4.2']=mc_rows
save_figure(fig,FIG,'fig_04_02_estimators')


OK: E4.2: SD średniej dla N=25 zgodne z σ/√N (8%)
OK: E4.2: średnia estymat wariancji N=25 (5%)
OK: E4.2: SD średniej dla N=100 zgodne z σ/√N (8%)
OK: E4.2: średnia estymat wariancji N=100 (5%)
OK: E4.2: SD średniej dla N=400 zgodne z σ/√N (8%)
OK: E4.2: średnia estymat wariancji N=400 (5%)
Figura: figures/generated/fig_04_02_estimators.png (+ SVG)


## E4.3 — gęstość, moc w prążku i kontrola Parsevala; rys. 4.3

Biały szum: `fs=200 Hz`, `SD=2 j.u.`, 400 realizacji dla `T=2 i 8 s`.
Porównanie: okno prostokątne, bez odjęcia średniej, `nfft=N`, oś jednostronna.
Seed: `BASE_SEED+3`. Dodatkowe kontrole: seed `BASE_SEED+30`, parzyste/nieparzyste
N i nfft, okno prostokątne/Hanna, DC zachowane/usunięte.

`P_bin = PSD * df` w panelu (a) jest mocą na siatce prostokątnego periodogramu.
Nie utożsamiamy tego z dowolnym wynikiem `scaling='spectrum'` dla innych okien.

![Rys. 4.3](../figures/generated/fig_04_03_density.png)


In [4]:
rng = np.random.default_rng(BASE_SEED+3)
fs, sigma = 200., 2.
fig,ax=plt.subplots(1,2,figsize=(9.2,3.7),layout='constrained')
rows=[]
for T,ls in [(2.,'-'),(8.,'--')]:
    N=round(fs*T)
    x=rng.normal(0,sigma,(400,N))
    f,p=signal.periodogram(x,fs=fs,window='boxcar',detrend=False,scaling='density',axis=-1)
    avg=p.mean(axis=0)
    ax[0].plot(f,avg*(fs/N),ls,lw=1.2,label=f'T={T:g} s')
    ax[1].plot(f,avg,ls,lw=1.2,label=f'T={T:g} s')
    level=avg[(f>5)&(f<80)].mean()
    check(f'E4.3: poziom PSD T={T:g} s (3%)',abs(level/(2*sigma**2/fs)-1)<.03)
    rows.append({'T_s':T,'N':N,'df_Hz':fs/N,'mean_density':float(level),'mean_bin_power':float(level*fs/N)})
ax[1].axhline(.04,color='black',ls=':',label='2σ²/fₛ = 0,04')
for a in ax:a.set(xlim=(5,60),xlabel='Częstotliwość [Hz]');a.legend()
ax[0].set(ylabel='Moc w prążku [j.u.²]',title='a) Inna szerokość przedziału')
ax[1].set(ylabel='PSD [j.u.²/Hz]',title='b) Ten sam poziom gęstości')
save_figure(fig,FIG,'fig_04_03_density')

rng=np.random.default_rng(BASE_SEED+30)
errors=[]
for N in [399,400]:
    x=2+rng.normal(0,1,N)
    m2=np.mean(x*x); var=np.var(x,ddof=0); mu=x.mean()
    check(f'E4.3: RMS²=wariancja_N+średnia², N={N}',np.isclose(m2,var+mu*mu))
    for window in ['boxcar','hann']:
        w=signal.get_window(window,N,fftbins=True)
        bw=fs*np.dot(w,w)/np.sum(w)**2
        for demean in [False,True]:
            for nfft in [N,2*N+1]:
                f,p,weighted=periodogram_density(x,fs,window,nfft,demean)
                f_ref,p_ref=signal.periodogram(x,fs=fs,window=w,nfft=nfft,
                    detrend='constant' if demean else False,scaling='density',return_onesided=True)
                check(f'E4.3: PSD ręczna=SciPy ({N},{window},{demean},{nfft})',np.allclose(p,p_ref,rtol=1e-11,atol=1e-12))
                check(f'E4.3: Parseval ({N},{window},{demean},{nfft})',np.isclose(p.sum()*fs/nfft,weighted,rtol=1e-12,atol=1e-12))
                _,spectrum=signal.periodogram(x,fs=fs,window=w,nfft=nfft,
                    detrend='constant' if demean else False,scaling='spectrum')
                check(f'E4.3: spectrum=density*B_w ({N},{window},{demean},{nfft})',np.allclose(spectrum,p*bw,rtol=1e-11,atol=1e-12))
                errors.append(float(abs(p.sum()*fs/nfft-weighted)))
RESULTS['E4.3']={'density_comparison':rows,'max_power_absolute_error':max(errors),'n_checks_combinations':16}


OK: E4.3: poziom PSD T=2 s (3%)
OK: E4.3: poziom PSD T=8 s (3%)
Figura: figures/generated/fig_04_03_density.png (+ SVG)
OK: E4.3: RMS²=wariancja_N+średnia², N=399
OK: E4.3: PSD ręczna=SciPy (399,boxcar,False,399)
OK: E4.3: Parseval (399,boxcar,False,399)
OK: E4.3: spectrum=density*B_w (399,boxcar,False,399)
OK: E4.3: PSD ręczna=SciPy (399,boxcar,False,799)
OK: E4.3: Parseval (399,boxcar,False,799)
OK: E4.3: spectrum=density*B_w (399,boxcar,False,799)
OK: E4.3: PSD ręczna=SciPy (399,boxcar,True,399)
OK: E4.3: Parseval (399,boxcar,True,399)
OK: E4.3: spectrum=density*B_w (399,boxcar,True,399)
OK: E4.3: PSD ręczna=SciPy (399,boxcar,True,799)
OK: E4.3: Parseval (399,boxcar,True,799)
OK: E4.3: spectrum=density*B_w (399,boxcar,True,799)
OK: E4.3: PSD ręczna=SciPy (399,hann,False,399)
OK: E4.3: Parseval (399,hann,False,399)
OK: E4.3: spectrum=density*B_w (399,hann,False,399)
OK: E4.3: PSD ręczna=SciPy (399,hann,False,799)
OK: E4.3: Parseval (399,hann,False,799)
OK: E4.3: spectrum=density*B_w 

## E4.4 — wariancja pojedynczego periodogramu; rys. 4.4

Po 2000 niezależnych realizacji białego szumu, `fs=200 Hz`, `SD=2 j.u.`,
`N=400,1600,6400`. Okno prostokątne, bez centrowania, `nfft=N`, częstotliwość
próby `25 Hz` wewnątrz pasma. Seed: `BASE_SEED+4`.
Wartości DFT przy 25 Hz wyznaczamy bezpośrednio jako iloczyn ze wzorcem;
kontrola porównuje je z pełnym periodogramem SciPy.

![Rys. 4.4](../figures/generated/fig_04_04_periodogram_variance.png)


In [5]:
rng=np.random.default_rng(BASE_SEED+4)
fs,sigma=200.,2.
theory=2*sigma**2/fs
fig,ax=plt.subplots(figsize=(8.6,3.9),layout='constrained')
rows=[]
for N,ls in [(400,'-'),(1600,'--'),(6400,':')]:
    values=[]
    basis=np.exp(-2j*np.pi*25*np.arange(N)/fs)
    for batch in range(20):
        x=rng.normal(0,sigma,(100,N))
        vals=2*np.abs(x@basis)**2/(fs*N)
        if batch==0:
            f,pp=signal.periodogram(x[0],fs=fs,window='boxcar',detrend=False,scaling='density')
            check(f'E4.4: DFT pojedynczej częstotliwości N={N}',np.isclose(vals[0],pp[np.argmin(abs(f-25))]))
        values.extend(vals.tolist())
    values=np.asarray(values)
    cv=values.std(ddof=1)/values.mean()
    hist,edges=np.histogram(values/theory,bins=np.linspace(0,7,65),density=False)
    ax.stairs(hist/(values.size*np.diff(edges)),edges,ls=ls,lw=1.6,label=f'N={N}, CV={cv:.2f}')
    check(f'E4.4: CV nie zanika, N={N}',.85<cv<1.15)
    check(f'E4.4: poziom średni N={N} (8%)',abs(values.mean()/theory-1)<.08)
    rows.append({'N':N,'mean_PSD':float(values.mean()),'SD_PSD':float(values.std(ddof=1)),'CV':float(cv)})
ax.axvline(1,color='black',ls='--',lw=1,label='Prawdziwy poziom / poziom odniesienia')
ax.set(xlabel='Periodogram / prawdziwa PSD [1]',ylabel='Gęstość [1]',xlim=(0,7),title='Dłuższa obserwacja nie stabilizuje pojedynczego prążka')
ax.legend()
RESULTS['E4.4']=rows
save_figure(fig,FIG,'fig_04_04_periodogram_variance')


OK: E4.4: DFT pojedynczej częstotliwości N=400
OK: E4.4: CV nie zanika, N=400
OK: E4.4: poziom średni N=400 (8%)
OK: E4.4: DFT pojedynczej częstotliwości N=1600
OK: E4.4: CV nie zanika, N=1600
OK: E4.4: poziom średni N=1600 (8%)
OK: E4.4: DFT pojedynczej częstotliwości N=6400
OK: E4.4: CV nie zanika, N=6400
OK: E4.4: poziom średni N=6400 (8%)
Figura: figures/generated/fig_04_04_periodogram_variance.png (+ SVG)


## E4.5 — Welch: czas segmentu i wariancja; rys. 4.5, tabela 4.1

`fs=200 Hz`, `T=32 s`, oscylacje `18 i 19.2 Hz`, amplitudy `1 i 0.8 j.u.`,
biały szum normalny `SD=1 j.u.`. Seed: `BASE_SEED+5`. Periodogram całego zapisu
i Welch: okresowe okno Hanna, centrowanie w analizowanym fragmencie/segmencie,
PSD jednostronna. Welch: `L=256,512,2048`, overlap 50%, `nfft=L`.
Jawna średnia periodogramów jest porównywana z `scipy.signal.welch`.

![Rys. 4.5](../figures/generated/fig_04_05_welch.png)


In [6]:
rng=np.random.default_rng(BASE_SEED+5)
fs=200.; t=np.arange(6400)/fs
x=np.cos(2*np.pi*18*t)+.8*np.cos(2*np.pi*19.2*t+.4)+rng.normal(0,1,t.size)
f,p=signal.periodogram(x,fs=fs,window=signal.get_window('hann',x.size,fftbins=True),detrend='constant',scaling='density')
fig,ax=plt.subplots(1,2,figsize=(9.8,4.0),layout='constrained')
for a in ax:a.semilogy(f,p,color='0.7',lw=.7,label='Periodogram całego zapisu')
rows=[]
for L,ls in [(256,'-'),(512,'--'),(2048,'-.')]:
    O=L//2
    fw,pw,starts=welch_manual(x,fs,L,O)
    fr,pr=signal.welch(x,fs=fs,window=signal.get_window('hann',L,fftbins=True),
                       nperseg=L,noverlap=O,nfft=L,detrend='constant',scaling='density',average='mean')
    check(f'E4.5: ręczny Welch=SciPy L={L}',np.allclose(pw,pr,rtol=1e-11,atol=1e-12))
    K=1+(len(x)-L)//(L-O)
    check(f'E4.5: liczba segmentów L={L}',len(starts)==K)
    for a in ax:a.semilogy(fw,pw,ls,lw=1.5,label=f'Welch L={L}, K={K}')
    rows.append({'L':L,'O':O,'K':K,'segment_s':L/fs,'df_Hz':fs/L,
                 'last_used_sample_exclusive':int(starts[-1]+L),'unused_tail':int(len(x)-starts[-1]-L)})
for a in ax:
    for freq in [18,19.2]:a.axvline(freq,color='black',ls=':',lw=.8)
    a.set(xlabel='Częstotliwość [Hz]',ylabel='PSD [j.u.²/Hz]')
ax[0].set(xlim=(3,45),ylim=(1e-5,12),title='a) Fluktuacje tła')
ax[1].set(xlim=(15.5,22),ylim=(1e-4,12),title='b) Zdolność rozdzielenia składowych')
ax[0].legend(loc='upper right')
RESULTS['E4.5']=rows
save_figure(fig,FIG,'fig_04_05_welch')


OK: E4.5: ręczny Welch=SciPy L=256
OK: E4.5: liczba segmentów L=256
OK: E4.5: ręczny Welch=SciPy L=512
OK: E4.5: liczba segmentów L=512
OK: E4.5: ręczny Welch=SciPy L=2048
OK: E4.5: liczba segmentów L=2048
Figura: figures/generated/fig_04_05_welch.png (+ SVG)


## E4.6 — nakładanie nie tworzy niezależnych danych; rys. 4.6

600 realizacji tego samego białego szumu, `N=6400`, `fs=200 Hz`, `SD=1 j.u.`.
Wszystkie warianty używają tych samych realizacji: `L=512`, okresowe okno Hanna,
centrowanie w segmencie, `nfft=L`; overlap `0,128,256,384,461` próbek.
Ocena przy `25 Hz`. Seed: `BASE_SEED+6`.

![Rys. 4.6](../figures/generated/fig_04_06_overlap.png)


In [7]:
rng=np.random.default_rng(BASE_SEED+6)
fs=200.;N=6400;L=512
draws=rng.normal(size=(600,N))
rows=[]
for O in [0,128,256,384,461]:
    vals=[]
    for chunk in np.array_split(draws,12):
        f,p=signal.welch(chunk,fs=fs,window=signal.get_window('hann',L,fftbins=True),
            nperseg=L,noverlap=O,nfft=L,detrend='constant',scaling='density',average='mean',axis=-1)
        vals.extend(p[:,np.argmin(abs(f-25))].tolist())
    vals=np.asarray(vals);K=len(segment_starts(N,L,O))
    rows.append({'overlap_samples':O,'overlap_pct':100*O/L,'K':K,
                 'CV':float(vals.std(ddof=1)/vals.mean()),'mean_PSD':float(vals.mean()),'independent_CV':float(1/np.sqrt(K))})
check('E4.6: zmniejszenie CV między overlap 0 i 50%', rows[2]['CV']<rows[0]['CV'])
check('E4.6: 90% nie daje CV niezależnych K segmentów',rows[-1]['CV']>1.2*rows[-1]['independent_CV'])
fig,ax=plt.subplots(figsize=(8.6,4.0),layout='constrained')
pct=[r['overlap_pct'] for r in rows]
ax.plot(pct,[r['CV'] for r in rows],'o-',label='Monte Carlo: obserwowane CV')
ax.plot(pct,[r['independent_CV'] for r in rows],'s--',label='1/√K: niezależne periodogramy')
for r in rows:ax.annotate(f"K={r['K']}",(r['overlap_pct'],r['CV']),xytext=(10 if r['overlap_pct']==0 else 0,10),textcoords='offset points',ha='left' if r['overlap_pct']==0 else 'center',fontsize=8)
ax.set(xlabel='Nakładanie [%]',ylabel='Współczynnik zmienności [1]',title='Liczba segmentów rośnie szybciej niż niezależna informacja',ylim=(0,.36))
ax.legend(loc='lower left')
RESULTS['E4.6']=rows
save_figure(fig,FIG,'fig_04_06_overlap')


OK: E4.6: zmniejszenie CV między overlap 0 i 50%
OK: E4.6: 90% nie daje CV niezależnych K segmentów
Figura: figures/generated/fig_04_06_overlap.png (+ SVG)


## E4.7 — lokalna stacjonarność EEG-podobnego sygnału; rys. 4.7

`fs=250 Hz`, `T=24 s`. Tło: kształtowanie FFT ~1/f, SD normalizowane dla całego
skończonego tła. `0–8 s`: SD=6 µV; `8–16 s`: dodatkowe 7 µV, 10 Hz;
`16–24 s`: mnożnik tła 1.65 i offset 3 µV. Seed: `BASE_SEED+7`.
To sygnał w całości syntetyczny, nie półsyntetyczny zapis rzeczywistego EEG.
Statystyki w blokach 2 s, SD z dzielnikiem N; PSD każdego stanu: Welch,
segment 2 s, Hann okresowy, 50% overlap, centrowanie w segmencie.

![Rys. 4.7](../figures/generated/fig_04_07_eeg_states.png)


In [8]:
rng=np.random.default_rng(BASE_SEED+7)
fs=250.; t=np.arange(6000)/fs
bg=colored_background(t.size,fs,rng)
scale=np.where(t>=16,1.65,1.)
x=6*scale*bg + 7*((t>=8)&(t<16))*np.sin(2*np.pi*10*t) + 3*(t>=16)
blocks=x.reshape(-1,500); centers=(np.arange(12)+.5)*2
fig,ax=plt.subplots(3,1,figsize=(9.3,8.0),layout='constrained')
ax[0].plot(t,x,lw=.65)
ax[0].set(xlabel='Czas [s]',ylabel='Amplituda [µV]',title='a) Trzy stany z zadanymi granicami')
ax[1].plot(centers,blocks.mean(axis=1),'o-',label='Średnia')
ax[1].plot(centers,blocks.std(axis=1,ddof=0),'s--',label='Odchylenie standardowe (N)')
ax[1].plot(centers,np.sqrt(np.mean(blocks**2,axis=1)),'^-.',label='RMS')
ax[1].set(xlabel='Środek okna [s]',ylabel='Wartość [µV]',title='b) Podobna średnia nie oznacza podobnej struktury')
ax[1].legend(ncol=3)
for a in ax[:2]:
    for boundary in [8,16]:a.axvline(boundary,color='black',ls=':',lw=1)
rows=[]
for a,b,label,ls in [(0,8,'0–8 s','-'),(8,16,'8–16 s','--'),(16,24,'16–24 s','-.'),(0,24,'Całe 24 s',':')]:
    block=x[int(a*fs):int(b*fs)]
    f,p=signal.welch(block,fs=fs,window=signal.get_window('hann',500,fftbins=True),nperseg=500,
                      noverlap=250,nfft=500,detrend='constant',scaling='density',average='mean')
    band=(f>=8)&(f<=12)
    rows.append({'interval_s':[a,b],'mean_uV':float(block.mean()),'SD_uV':float(block.std(ddof=0)),
                 'band_8_12_uV2':float(p[band].sum()*(f[1]-f[0]))})
    ax[2].semilogy(f,p,ls,lw=1.5,label=label)
ax[2].set(xlim=(2,30),xlabel='Częstotliwość [Hz]',ylabel='PSD [µV²/Hz]',title='c) Globalna PSD nie lokalizuje rytmu w czasie')
ax[2].legend(ncol=4)
check('E4.7: tło normalizowane do SD=1 i średniej 0',np.isclose(bg.std(),1) and abs(bg.mean())<1e-12)
check('E4.7: rytm zwiększa moc 8–12 Hz w środkowym stanie',rows[1]['band_8_12_uV2']>rows[0]['band_8_12_uV2'])
RESULTS['E4.7']=rows
save_figure(fig,FIG,'fig_04_07_eeg_states')

# Kontrola pomocnicza na osobnym zapisie bez zmiany stanu: stała oscylacja 10,2 Hz.
rng=np.random.default_rng(BASE_SEED+70)
stable=7*colored_background(t.size,fs,rng)+8*np.sin(2*np.pi*10.2*t+.4)+rng.normal(0,2,t.size)
stable_rows=[]
for seconds in [2,4,8]:
    L=int(seconds*fs)
    f,p=signal.welch(stable,fs=fs,window=signal.get_window('hann',L,fftbins=True),
        nperseg=L,noverlap=L//2,nfft=L,detrend='constant',scaling='density',average='mean')
    mask=(f>=8)&(f<=12); peak=f[mask][np.argmax(p[mask])]
    check(f'E4.7: obecność maksimum około 10,2 Hz, segment {seconds}s',abs(peak-10.2)<=.5)
    stable_rows.append({'segment_s':seconds,'K':len(segment_starts(t.size,L,L//2)),
                        'peak_Hz':float(peak),'band_uV2':float(p[mask].sum()*(f[1]-f[0]))})
RESULTS['E4.7_stable_sensitivity']=stable_rows
print('Czułość wyniku stabilnego EEG-podobnego zapisu:',stable_rows)


OK: E4.7: tło normalizowane do SD=1 i średniej 0
OK: E4.7: rytm zwiększa moc 8–12 Hz w środkowym stanie
Figura: figures/generated/fig_04_07_eeg_states.png (+ SVG)
OK: E4.7: obecność maksimum około 10,2 Hz, segment 2s
OK: E4.7: obecność maksimum około 10,2 Hz, segment 4s
OK: E4.7: obecność maksimum około 10,2 Hz, segment 8s
Czułość wyniku stabilnego EEG-podobnego zapisu: [{'segment_s': 2, 'K': 23, 'peak_Hz': 10.0, 'band_uV2': 37.38283478014055}, {'segment_s': 4, 'K': 11, 'peak_Hz': 10.25, 'band_uV2': 37.4780689482818}, {'segment_s': 8, 'K': 5, 'peak_Hz': 10.25, 'band_uV2': 36.89010066542565}]


## E4.8 — autokorelacja a okres; rys. 4.8

`fs=200 Hz`, `T=20 s`, seed `BASE_SEED+8`; szum normalny SD=1 j.u.,
sinusoida 2.5 Hz + szum SD=0.85 j.u.; ciąg impulsów: nominalny odstęp 0.92 s,
SD jitteru odstępów 0.025 s, szerokość Gaussa 0.05 s, szum SD=0.28 j.u.,
seed `BASE_SEED+80`. Odjęcie średniej, dzielnik N, normalizacja przez C[0].
Zakres okresu 0.5–1.4 s; wybór najwyższego lokalnego maksimum o prominence >=0.08.
Nie jest to ogólny detektor rytmu fizjologicznego.

![Rys. 4.8](../figures/generated/fig_04_08_autocorrelation.png)


In [9]:
rng=np.random.default_rng(BASE_SEED+8)
fs=200.; t=np.arange(4000)/fs
white=rng.normal(size=t.size)
sine=np.sin(2*np.pi*2.5*t+.4)+rng.normal(0,.85,t.size)
tp,pulses,events,clean=pulse_train(fs,20.,.92,.025,.28,BASE_SEED+80)
fig,ax=plt.subplots(3,2,figsize=(9.5,7.5),layout='constrained')
for row,(label,x) in enumerate([('Biały szum',white),('2,5 Hz + szum',sine),('Ciąg quasiokresowych impulsów',pulses)]):
    k,rho,c=autocorrelation(x,'N')
    check(f'E4.8: rho[0]=1 i |rho|<=1, {label}',np.isclose(rho[0],1) and np.max(abs(rho))<1+1e-12)
    ax[row,0].plot(t,x,lw=.7)
    ax[row,0].set(xlim=(0,6),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title=label)
    ax[row,1].plot(k/fs,rho,lw=1.2)
    ax[row,1].set(xlim=(0,3),ylim=(-.7,1.05),xlabel='Opóźnienie [s]',ylabel='Autokorelacja [1]')
    if row==2:
        low,high=int(.5*fs),int(1.4*fs)
        peaks,_=signal.find_peaks(rho[low:high+1],prominence=.08)
        check('E4.8: istnieje kandydat okresu',len(peaks)>0)
        candidates=peaks+low
        k0=int(candidates[np.argmax(rho[candidates])])
        ax[row,1].axvspan(.5,1.4,alpha=.08,color='#25805D')
        ax[row,1].axvline(k0/fs,color='black',ls='--',lw=1,label=f'T={k0/fs:.3f} s')
        ax[row,1].legend()
        RESULTS['E4.8']={'k0':k0,'period_s':k0/fs,'frequency_Hz':fs/k0,'nominal_period_s':.92,
                         'realized_mean_interval_s':float(np.diff(events).mean()),'n_events':len(events)}
        check('E4.8: okres bliski konstrukcji, tolerancja 0,06 s',abs(k0/fs-.92)<.06)
save_figure(fig,FIG,'fig_04_08_autocorrelation')


OK: E4.8: rho[0]=1 i |rho|<=1, Biały szum
OK: E4.8: rho[0]=1 i |rho|<=1, 2,5 Hz + szum
OK: E4.8: rho[0]=1 i |rho|<=1, Ciąg quasiokresowych impulsów
OK: E4.8: istnieje kandydat okresu
OK: E4.8: okres bliski konstrukcji, tolerancja 0,06 s
Figura: figures/generated/fig_04_08_autocorrelation.png (+ SVG)


## E4.9 — dzielnik, duże opóźnienia i Wiener–Chinczyn; rys. 4.9

Krótki biały szum, `N=120`, `fs=200 Hz`, `SD=1 j.u.`, seed `BASE_SEED+9`.
Porównanie dzielników N i N−k po odjęciu średniej. Korelacja z dzielnikiem N−k
podzielona przez C[0] nie musi być ograniczona do [-1,1].
Kontrola Wienera–Chinczyna: seed `BASE_SEED+90`, osobny zapis N=3200,
składowe 7 i 14 Hz, amplitudy 1.2 i 0.7 j.u., szum SD=1 j.u.;
centrowanie, prostokątne okno, autokorelacja **kołowa** z dzielnikiem N,
dwustronna PSD `DFT(C)/fs`, następnie prawidłowe składanie jednostronne.

![Rys. 4.9](../figures/generated/fig_04_09_acf_edges.png)


In [10]:
rng=np.random.default_rng(BASE_SEED+9)
fs=200.;x=rng.normal(size=120)
k,rn,cn=autocorrelation(x,'N');_,ru,cu=autocorrelation(x,'N-k')
fig,ax=plt.subplots(2,1,figsize=(8.6,5.7),sharex=True,layout='constrained')
ax[0].plot(k/fs,rn,'-',label='Dzielnik N')
ax[0].plot(k/fs,ru,'--',label='Dzielnik N−k')
ax[0].axhline(1,color='black',ls=':',lw=.8);ax[0].axhline(-1,color='black',ls=':',lw=.8)
ax[0].set(ylabel='C[k] / C[0] [1]',title='a) Korekta liczby par wzmacnia także fluktuacje')
ax[0].legend()
ax[1].plot(k/fs,len(x)-k,color='black')
ax[1].set(xlabel='Opóźnienie [s]',ylabel='Liczba par',title='b) Informacja dostępna dla danego przesunięcia')
check('E4.9: relacja między dzielnikami',np.allclose(cu,cn*len(x)/(len(x)-k)))
RESULTS['E4.9']={'max_abs_rho_NminusK':float(max(abs(ru))),'pairs_at_last_lag':1}
save_figure(fig,FIG,'fig_04_09_acf_edges')

rng=np.random.default_rng(BASE_SEED+90)
N=3200; t=np.arange(N)/fs
x=1.2*np.sin(2*np.pi*7*t)+.7*np.sin(2*np.pi*14*t+.6)+rng.normal(size=N)
x-=x.mean()
X=np.fft.fft(x)
circular=np.fft.ifft(abs(X)**2).real/N
p2=np.fft.fft(circular).real/fs
p_from_c=one_sided(p2,N)
f,p,_=periodogram_density(x,fs,'boxcar',demean=False)
err=float(np.max(abs(p-p_from_c)))
check('E4.9: Wiener–Chinczyn ze skalowaniem w Hz',np.allclose(p,p_from_c,rtol=1e-10,atol=1e-12))
RESULTS['E4.9']['WC_max_abs_error']=err
print('Maksymalna różnica PSD i DFT autokorelacji kołowej:',err)


OK: E4.9: relacja między dzielnikami
Figura: figures/generated/fig_04_09_acf_edges.png (+ SVG)
OK: E4.9: Wiener–Chinczyn ze skalowaniem w Hz
Maksymalna różnica PSD i DFT autokorelacji kołowej: 1.7763568394002505e-15


## E4.10 — znak opóźnienia i wieloznaczność; rys. 4.10

Impuls asymetryczny: `fs=500 Hz`, `T=2 s`, opóźnienie `18 próbek=36 ms`,
drugi kanał pomnożony przez 1.7; niezależny szum SD=0.025 j.u. Seed `BASE_SEED+10`.
Centrowanie całych kanałów, normalizacja energią części wspólnej; min. 100 par.
Zakres ±100 ms. Dodatni lag oznacza, że y jest opóźniony względem x.
Kontrole: bezszumowa znana para, zamiana kanałów, dodatnia skala,
porównanie z `scipy.signal.correlate(y0,x0)` i wyrównanie.

Para okresowa: `fs=200 Hz`, `T=10 s`, podstawowa 1.5 Hz i druga harmoniczna,
opóźnienie 24 próbki (0.12 s), szum SD=0.1, seed `BASE_SEED+100`.

![Rys. 4.10](../figures/generated/fig_04_10_cross_correlation.png)


In [11]:
rng=np.random.default_rng(BASE_SEED+10)
fs=500.;t=np.arange(1000)/fs
s=np.exp(-.5*((t-.75)/.045)**2)-.55*np.exp(-.5*((t-.86)/.075)**2)+.28*np.exp(-.5*((t-1.03)/.12)**2)
x=s+rng.normal(0,.025,len(t)); y=1.7*delay_zeros(s,18)+rng.normal(0,.025,len(t))
lags,rxy,pairs=cross_correlation(x,y,50,100)
_,ryx,_=cross_correlation(y,x,50,100)
kh=int(lags[np.nanargmax(rxy)]); kh_rev=int(lags[np.nanargmax(ryx)])
check('E4.10: zaszumione opóźnienie w granicach 1 próbki',abs(kh-18)<=1)
check('E4.10: zamiana kanałów odwraca znak',kh_rev==-kh and np.allclose(ryx,rxy[::-1]))
_,scaled,_=cross_correlation(x,3*y,50,100)
check('E4.10: dodatnia skala całego kanału nie zmienia rho',np.allclose(rxy,scaled))
kl,rl,_=cross_correlation(s,delay_zeros(s,18),50,100)
check('E4.10: znana bezszumowa para +18 próbek',kl[np.nanargmax(rl)]==18)
# Niezależna kontrola kolejności argumentów SciPy oraz wartości po normalizacji.
x0,y0=x-x.mean(),y-y.mean()
raw=signal.correlate(y0,x0,mode='full',method='direct')
ref_lags=signal.correlation_lags(len(y0),len(x0),mode='full')
check('E4.10: znak maksimum nieznormalizowanej korelacji SciPy',abs(int(ref_lags[np.argmax(raw)])-18)<=1)
alignment_mse=float(np.mean((x[:-kh]-y[kh:]/1.7)**2))
unshifted_mse=float(np.mean((x-y/1.7)**2))
check('E4.10: wyrównanie zmniejsza błąd porównania',alignment_mse<unshifted_mse)

fig,ax=plt.subplots(3,1,figsize=(9.0,8.0),layout='constrained')
ax[0].plot(t,x,lw=1,label='x');ax[0].plot(t,y,'--',lw=1,label='y (×1,7)')
ax[0].set(xlim=(.55,1.15),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Znany impuls i opóźniony kanał')
ax[0].legend()
ax[1].plot(lags/fs*1000,rxy,label='ρxy');ax[1].plot(lags/fs*1000,ryx,'--',label='ρyx')
for lag in [18,-18]:ax[1].axvline(lag/fs*1000,color='black',ls=':',lw=.9)
ax[1].set(xlabel='Opóźnienie [ms]',ylabel='Korelacja [1]',title=f'b) Maksima: {kh} i {kh_rev} próbek')
ax[1].legend()

rng=np.random.default_rng(BASE_SEED+100)
fsp=200.;tp=np.arange(2000)/fsp
sp=np.sin(2*np.pi*1.5*tp)+.35*np.sin(2*np.pi*3*tp+.4)
xp=sp+rng.normal(0,.1,tp.size);yp=.9*delay_zeros(sp,24)+rng.normal(0,.1,tp.size)
kp,rp,_=cross_correlation(xp,yp,440,300)
ax[2].plot(kp/fsp,rp,lw=1.2)
ax[2].axvspan(0,.2,alpha=.13,color='#25805D',label='Dopuszczalny zakres przykładu')
ax[2].axvline(.12,color='black',ls='--',lw=1,label='Znane opóźnienie 0,12 s')
ax[2].set(xlabel='Opóźnienie [s]',ylabel='Korelacja [1]',title='c) Okresowość tworzy konkurujące maksima')
ax[2].legend(loc='lower right')
RESULTS['E4.10']={'delay_samples':kh,'delay_ms':kh/fs*1000,'reversed_samples':kh_rev,
    'true_delay_samples':18,'true_delay_ms':36.,'aligned_MSE':alignment_mse,'unshifted_MSE':unshifted_mse,
    'min_pairs_used':int(pairs.min()),'max_rho':float(np.nanmax(rxy))}
save_figure(fig,FIG,'fig_04_10_cross_correlation')


OK: E4.10: zaszumione opóźnienie w granicach 1 próbki
OK: E4.10: zamiana kanałów odwraca znak
OK: E4.10: dodatnia skala całego kanału nie zmienia rho
OK: E4.10: znana bezszumowa para +18 próbek
OK: E4.10: znak maksimum nieznormalizowanej korelacji SciPy
OK: E4.10: wyrównanie zmniejsza błąd porównania
Figura: figures/generated/fig_04_10_cross_correlation.png (+ SVG)


## E4.11 — średnia synchroniczna i SNR; rys. 4.11

`fs=500 Hz`, 128 epok, `t=-0.1...0.498 s`, `SD szumu=12 µV`, seed `BASE_SEED+11`.
Odpowiedź: suma trzech Gaussianów (2.2, -4, 3 µV; centra 65,105,165 ms;
szerokości 18,24,38 ms). Linia bazowa [-100,0) ms; okno odpowiedzi [40,220] ms.
SNR amplitudowy = połowa ptp **znanej** odpowiedzi w tym oknie / SD średniej
linii bazowej (dzielnik N−1). Ta definicja SNR nie jest dostępną bezpośrednio
miarą rzeczywistego badania, gdy odpowiedź odniesienia jest nieznana.

![Rys. 4.11](../figures/generated/fig_04_11_averaging.png)


In [12]:
rng=np.random.default_rng(BASE_SEED+11)
fs=500.;te=np.arange(300)/fs-.1
truth=evoked_response(te);noise=rng.normal(0,12,(128,te.size));epochs=truth[None,:]+noise
baseline=(te>=-.1)&(te<0);response=(te>=.04)&(te<=.22)
amplitude=np.ptp(truth[response])/2
Ms=np.array([1,2,4,8,16,32,64,128])
rows=[]
for M in Ms:
    avg=epochs[:M].mean(axis=0)
    sd=float(avg[baseline].std(ddof=1))
    rows.append({'M':int(M),'SD_baseline_uV':sd,'SD_theory_uV':float(12/np.sqrt(M)),
                 'amplitude_SNR':float(amplitude/sd),
                 'RMS_error_uV':float(np.sqrt(np.mean((avg-truth)**2)))})
fig,ax=plt.subplot_mosaic([['single','sd'],['average','sd']],figsize=(10.0,6.3),layout='constrained')
ax['single'].plot(te*1000,epochs[0],lw=.7,label='M=1')
ax['single'].plot(te*1000,truth,color='black',lw=1.5,label='Odpowiedź odniesienia')
for M,ls in [(16,'--'),(128,'-')]:
    ax['average'].plot(te*1000,epochs[:M].mean(axis=0),ls,lw=1.3,label=f'M={M}')
ax['average'].plot(te*1000,truth,color='black',lw=1.5,label='Odpowiedź odniesienia')
for key in ['single','average']:
    ax[key].axvline(0,color='black',ls=':',lw=.8)
    ax[key].set(xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [µV]',xlim=(-100,350))
    ax[key].legend(fontsize=7)
ax['single'].set(title='a) Jedna epoka: odpowiedź pod szumem')
ax['average'].set(title='b) Średnie: powiększona skala amplitudy',ylim=(-8,8))
ax['sd'].loglog(Ms,[r['SD_baseline_uV'] for r in rows],'o-',label='SD linii bazowej')
ax['sd'].loglog(Ms,12/np.sqrt(Ms),'--',color='black',label='12 / √M')
ax['sd'].set(xlabel='Liczba epok M',ylabel='Odchylenie standardowe [µV]',title='c) Spadek szumu i jego fluktuacje')
ax['sd'].legend()
check('E4.11: 128 epok zmniejsza RMS błędu względem 1 epoki',rows[-1]['RMS_error_uV']<rows[0]['RMS_error_uV']/5)
# Kontrola niezależności między epokami: wariancja średniej w ustalonej próbce
# jest badana w osobnym Monte Carlo, bez utożsamiania czasu z zespołem realizacji.
rng_check=np.random.default_rng(BASE_SEED+110)
fixed_point=rng_check.normal(0,12,(4000,128)).mean(axis=1)
mc_variance=float(fixed_point.var(ddof=1))
check('E4.11: wariancja szumu średniej = σ²/M (6%)',abs(mc_variance/(144/128)-1)<.06)
even=epochs[::2].mean(axis=0);odd=epochs[1::2].mean(axis=0)
RESULTS['E4.11']={'averages':rows,'A_reference_uV':float(amplitude),
    'MC_variance_of_mean_uV2':mc_variance,'even_odd_RMS_difference_uV':float(np.sqrt(np.mean((even-odd)**2))),
    'epochs_recorded':128,'epochs_rejected':0,'epochs_used':128,'baseline_samples':int(baseline.sum())}
print('Redukcja SD i SNR amplitudowy:',rows)
print('RMS różnicy średnich parzystej i nieparzystej:',RESULTS['E4.11']['even_odd_RMS_difference_uV'])
save_figure(fig,FIG,'fig_04_11_averaging')


OK: E4.11: 128 epok zmniejsza RMS błędu względem 1 epoki
OK: E4.11: wariancja szumu średniej = σ²/M (6%)
Redukcja SD i SNR amplitudowy: [{'M': 1, 'SD_baseline_uV': 12.08353534671261, 'SD_theory_uV': 12.0, 'amplitude_SNR': 0.24127925829044652, 'RMS_error_uV': 11.844551689274804}, {'M': 2, 'SD_baseline_uV': 9.080968107260887, 'SD_theory_uV': 8.48528137423857, 'amplitude_SNR': 0.3210567872879165, 'RMS_error_uV': 8.501705795580952}, {'M': 4, 'SD_baseline_uV': 6.710449353442096, 'SD_theory_uV': 6.0, 'amplitude_SNR': 0.4344726101666673, 'RMS_error_uV': 6.016276261002324}, {'M': 8, 'SD_baseline_uV': 4.142973860656746, 'SD_theory_uV': 4.242640687119285, 'amplitude_SNR': 0.7037231090613361, 'RMS_error_uV': 4.101720718442901}, {'M': 16, 'SD_baseline_uV': 2.787942775165899, 'SD_theory_uV': 3.0, 'amplitude_SNR': 1.0457554839186833, 'RMS_error_uV': 2.8769360654386213}, {'M': 32, 'SD_baseline_uV': 1.8248694624202093, 'SD_theory_uV': 2.1213203435596424, 'amplitude_SNR': 1.597652054582885, 'RMS_error_

## E4.12 — jitter, morfologia i artefakt; rys. 4.12

Ten sam szum co w E4.11 we wszystkich wariantach; oddzielny seed modelu `BASE_SEED+12`.
Jitter całkowity jednostajny −8…+8 próbek (±16 ms); bez kołowego zawijania.
Morfologia: skala amplitudy N(1,0.22²) obcięta [0.3,1.7], skala szerokości
N(1,0.25²) obcięta [0.55,1.5]. Artefakt: Gauss 5.5 µV, centrum 12 ms, szerokość 8 ms.
Wykres pokazuje także średnie czystych odpowiedzi w każdym wariancie;
pozwala oddzielić obciążenie morfologii od resztkowego szumu.

![Rys. 4.12](../figures/generated/fig_04_12_averaging_failures.png)


In [13]:
rng=np.random.default_rng(BASE_SEED+12)
clean_ideal=np.broadcast_to(truth,(128,len(te))).copy()
jitters=rng.integers(-8,9,128)
clean_jitter=np.array([evoked_response(te-j/fs) for j in jitters])
amplitudes=np.clip(rng.normal(1,.22,128),.3,1.7)
widths=np.clip(rng.normal(1,.25,128),.55,1.5)
clean_morph=np.array([evoked_response(te,a,w) for a,w in zip(amplitudes,widths)])
artifact=5.5*np.exp(-.5*((te-.012)/.008)**2)
clean_artifact=clean_ideal+artifact[None,:]
variants=[('Idealna synchronizacja',clean_ideal),('Jitter ±16 ms',clean_jitter),
          ('Zmienna morfologia',clean_morph),('Artefakt po zdarzeniu',clean_artifact)]
fig,ax=plt.subplots(2,2,figsize=(9.6,6.8),sharex=True,sharey=True,layout='constrained')
rows=[]
for a,(label,clean_epochs) in zip(ax.flat,variants):
    clean_mean=clean_epochs.mean(axis=0)
    average=(clean_epochs+noise).mean(axis=0)
    a.plot(te*1000,average,lw=1.2,label='Średnia 128 epok')
    a.plot(te*1000,clean_mean,'--',lw=1.5,label='Średnia bez szumu')
    a.plot(te*1000,truth,':',color='black',lw=1.5,label='Odpowiedź odniesienia')
    a.axvline(0,color='black',lw=.8,ls=':')
    a.set(title=label,xlim=(-50,300),ylim=(-5.5,7.0),xlabel='Czas względem zdarzenia [ms]',ylabel='Amplituda [µV]')
    rows.append({'variant':label,'clean_mean_RMS_deviation_uV':float(np.sqrt(np.mean((clean_mean-truth)**2))),
                 'clean_mean_ptp_uV':float(np.ptp(clean_mean[response]))})
ax[0,0].legend(loc='upper right',fontsize=7)
check('E4.12: zsynchronizowany artefakt nie znika w średniej',np.allclose(clean_artifact.mean(axis=0)-truth,artifact))
check('E4.12: jitter zmienia czystą średnią',rows[1]['clean_mean_RMS_deviation_uV']>.01)
check('E4.12: jitter obniża ptp czystej odpowiedzi',rows[1]['clean_mean_ptp_uV']<rows[0]['clean_mean_ptp_uV'])
RESULTS['E4.12']=rows
save_figure(fig,FIG,'fig_04_12_averaging_failures')


OK: E4.12: zsynchronizowany artefakt nie znika w średniej
OK: E4.12: jitter zmienia czystą średnią
OK: E4.12: jitter obniża ptp czystej odpowiedzi
Figura: figures/generated/fig_04_12_averaging_failures.png (+ SVG)


## Wynik wykonania i zakres kontroli

Ostatnia komórka zapisuje liczby używane w audycie do `data/ch04_results.json`,
a wersje środowiska do `chapter_notes/ch04_environment.json`. Asercje kontrolują
skalowanie, znak, zgodność definicji i referencje symulacyjne. Nie zastępują
walidacji na danych rzeczywistych ani audytu bibliograficznego.
Ustawienia figur i pomocnicze funkcje są częścią pakietu; samo przeniesienie pliku
IPYNB bez `src/` nie wystarcza do odtworzenia obliczeń.


In [14]:
expected=[f'fig_04_{i:02d}_' for i in range(1,13)]
for prefix in expected:
    for ext in ['png','svg']:
        check(f'Figura {prefix}*.{ext} istnieje',len(list(FIG.glob(prefix+'*.'+ext)))==1)
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS,'experiment_count':12,
                  'figure_count':12,'external_data':False}
(ROOT/'data').mkdir(exist_ok=True)
(ROOT/'chapter_notes').mkdir(exist_ok=True)
(ROOT/'data/ch04_results.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(ROOT/'chapter_notes/ch04_environment.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Wszystkie eksperymenty zakończone. Liczba kontroli:',len(CHECKS))


OK: Figura fig_04_01_*.png istnieje
OK: Figura fig_04_01_*.svg istnieje
OK: Figura fig_04_02_*.png istnieje
OK: Figura fig_04_02_*.svg istnieje
OK: Figura fig_04_03_*.png istnieje
OK: Figura fig_04_03_*.svg istnieje
OK: Figura fig_04_04_*.png istnieje
OK: Figura fig_04_04_*.svg istnieje
OK: Figura fig_04_05_*.png istnieje
OK: Figura fig_04_05_*.svg istnieje
OK: Figura fig_04_06_*.png istnieje
OK: Figura fig_04_06_*.svg istnieje
OK: Figura fig_04_07_*.png istnieje
OK: Figura fig_04_07_*.svg istnieje
OK: Figura fig_04_08_*.png istnieje
OK: Figura fig_04_08_*.svg istnieje
OK: Figura fig_04_09_*.png istnieje
OK: Figura fig_04_09_*.svg istnieje
OK: Figura fig_04_10_*.png istnieje
OK: Figura fig_04_10_*.svg istnieje
OK: Figura fig_04_11_*.png istnieje
OK: Figura fig_04_11_*.svg istnieje
OK: Figura fig_04_12_*.png istnieje
OK: Figura fig_04_12_*.svg istnieje
Wszystkie eksperymenty zakończone. Liczba kontroli: 123
